<a href="https://colab.research.google.com/github/Ysabellyysa/Python/blob/main/Detec%C3%A7%C3%A3o_de_Emo%C3%A7%C3%B5es_em_Videos_Aula.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detecção de Emoções em Videos

### Etapa 1 - Importando as bibliotecas

In [1]:
import cv2 # biblioteca de visão computacional
import numpy as np
import time #funções de tempo e pausas
import pandas as pd
import matplotlib.pyplot as plt # criação de gráficos
from google.colab.patches import cv2_imshow
import zipfile # manipulação de arquivos compactados
cv2.__version__ # mostra a versão do OpenCV

'4.14.0'

In [2]:
%tensorflow_version 2.x
import tensorflow # biblioteca para redes neurais
tensorflow.__version__ # mostra a versão do TensorFlow

Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.


'2.20.0'

### Etapa 2 - Conectando com o Drive e acessando os arquivos

In [3]:
from google.colab import drive
drive.mount('/content/gdrive') # conecta o google drive ao notebook
path = "/content/gdrive/My Drive/Material.zip" # define o caminho do arquivo zip
zip_object = zipfile.ZipFile(file=path, mode="r")
zip_object.extractall("./") # extrai todo o conteúdo da pasta atual
zip_object.close

Mounted at /content/gdrive


<bound method ZipFile.close of <zipfile.ZipFile filename='/content/gdrive/My Drive/Material.zip' mode='r'>>

### Etapa 3 - Carregando o modelo

In [ ]:
from tensorflow.keras.models import load_model # importa a função para carregar modelos
model = load_model("Material/modelo_02_expressoes.h5") # carrega o arquivo

### Etapa 4 - Carregando o vídeo

In [ ]:
arquivo_video = "Material/Videos/video_teste04.mp4" #define o caminho do arquivo de vídeo
cap = cv2.VideoCapture(arquivo_video) #abre o vídeo para processamento

conectado, video = cap.read()
print(conectado, video.shape) # mostra se o vídeo carregou e o tamanho da imagem

### Etapa 5 - Redimensionando o tamanho (opcional)

In [ ]:
redimensionar = True #ativa ou desativa o ajuste de tamanho
largura_maxima = 600 #define o limite de largura para o vídeo

if (redimensionar and video.shape[1] > largura_maxima):
    proporcao = video.shape[1] / video.shape[0] # calcula a proporção entre largura e altura
    video_largura = largura_maxima # define a nova largura fixa
    video_altura = int(video_largura / proporcao)
else:
    video_largura = video.shape[1] # mantém a largura original
    video_altura = video.shape[0] # mantém a altura original

### Etapa 6 - Definindo as configurações do vídeo

In [ ]:
# Nome do arquivo que será salvo
nome_arquivo = 'resultado_video_teste04.avi'
fourcc = cv2.VideoWriter_fourcc(*'XVID')
fps = 24

saida_video = cv2.VideoWriter(nome_arquivo, fourcc, fps, (video_largura, video_altura))

###Etapa 7 - Processamento do vídeo e gravação do resultado

In [ ]:
from tensorflow.keras.preprocessing.image import img_to_array #importa conversor de imagem para array

haarcascade_faces = 'Material/haarcascade_frontalface_default.xml'
fonte_pequena, fonte_media = 0.4, 0.7 # define tamanhos de fonte
fonte = cv2.FONT_HERSHEY_SIMPLEX # define o tipo da fonte
expressoes = ["Raiva", "Nojo", "Medo", "Feliz", "Triste", "Surpreso", "Neutro"] # lista de emoções

while (cv2.waitKey(1) < 0): # inicia loop enquanto o vídeo rodar
    conectado, frame = cap.read()

    if not conectado: #para o loop se o vídeo acabar
        break

    t = time.time()

    if redimensionar: #verifica se deve ajustar o tamanho
        frame = cv2.resize(frame, (video_largura, video_altura)) #redimensiona o quadro atual

    face_cascade = cv2.CascadeClassifier(haarcascade_faces)
    cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) # converte a imagem para cinza
    faces = face_cascade.detectMultiScale(cinza, scaleFactor=1.2, minNeighbors=5, minSize=(30,30)) # detecta rostos

    if len(faces) > 0:
        for (x, y, w, h) in faces: # percorre cada rosto detectado
            frame = cv2.rectangle(frame, (x, y), (x + w, y + h + 10), (255, 50, 50), 2) # desenha o retângulo
            roi = cinza[y:y + h, x:x + w] # recorta apenas o rosto
            roi = cv2.resize(roi, (48, 48))
            roi = roi.astype("float") / 255.0 # normaliza os valores dos pixels
            roi = img_to_array(roi) #transforma em formato de array
            roi = np.expand_dims(roi, axis=0) #adiciona dimensão extra para o modelo

            result = model.predict(roi)[0] #faz a previsão da emoção
            print(result) #imprime as probabilidades no console

            if result is not None:
                resultado = np.argmax(result)
                cv2.putText(frame, expressoes[resultado], (x, y - 10), fonte, fonte_media, (255, 255, 255), 1, cv2.LINE_AA) # escreve a emoção no vídeo

    cv2.putText(frame, " frame processado em {:.2f} segundos".format(time.time() - t), (20, video_altura - 20), fonte, fonte_pequena, (250, 250, 250), 0, cv2.LINE_AA) #mostra tempo de execução

    cv2_imshow(frame)
    saida_video.write(frame) # salva o quadro no arquivo de saída

print("Terminou") # avisa que o processo finalizou
saida_video.release() # fecha o arquivo de vídeo salvo
cv2.destroyAllWindows() # fecha as janelas abertas